# Hospital Admission Forecasting

## Project Overview

Forecasts **daily hospital admissions** over a 14-day horizon using synthetic data spanning 2 years with weekly patterns, flu season effects, and holiday dips.

**Models**: Naive, LazyPredict, FLAML, StatsForecast. **Horizon**: 14 days.

## Learning Objectives

1. Understand time-series patterns (trend, seasonality, noise)
2. Build naive and seasonal-naive baselines
3. Engineer lag and rolling features for tabular ML
4. Use LazyPredict for quick regression benchmarking
5. Apply FLAML AutoML (excluding XGBoost due to compatibility)
6. Use StatsForecast (AutoARIMA, AutoETS, SeasonalNaive)
7. Compare all approaches with MAE / RMSE / MAPE

## Problem Statement

Given historical daily admission counts, predict the next 14 days. This drives bed allocation, staffing, and resource planning.

## Why This Project Matters

Hospital capacity planning saves lives. Over-crowding leads to longer waits, worse outcomes, and diversion of ambulances. Under-utilization wastes resources.

## Dataset Overview

Synthetic dataset:
- 730 days (2 years) of daily hospital admissions
- Weekly pattern (higher weekdays, lower weekends for elective)
- Flu season surge (Jan-Mar)
- Holiday dips (Christmas, Thanksgiving)
- Gradual upward trend

| Column | Description |
|--------|-------------|
| `date` | Date |
| `admissions` | Daily hospital admission count |

## Dataset Source and License Notes

Synthetically generated for educational purposes. No external download required.

## Environment Setup

In [1]:
import subprocess, importlib
for pkg in ['numpy', 'pandas', 'matplotlib', 'scikit-learn', 'statsforecast', 'flaml', 'lazypredict']:
    try:
        importlib.import_module(pkg.replace('-', '_').split('[')[0])
    except ImportError:
        subprocess.check_call(['pip', 'install', '-q', pkg])
print("All packages ready.")

C:\Users\ahmad\AppData\Local\Programs\Python\Python313\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.0.post2)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


All packages ready.


## Imports

In [2]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
from sklearn.metrics import mean_absolute_error, mean_squared_error
print("Imports complete.")

Imports complete.


## Configuration / Constants

In [3]:
SEED = 42
N_POINTS = 730
HORIZON = 14
VAL_SIZE = 14
TEST_SIZE = 14
SEASON_LENGTH = 7
FREQ = 'D'
TARGET = 'admissions'
np.random.seed(SEED)
print(f"Config: {N_POINTS} points, horizon={HORIZON}, season={SEASON_LENGTH}")

Config: 730 points, horizon=14, season=7


## Dataset Generation

In [4]:
dates = pd.date_range(start='2022-01-01', periods=N_POINTS, freq='D')
t = np.arange(N_POINTS)

base = 200 + 0.15 * t
weekly = np.zeros(N_POINTS)
for i in range(N_POINTS):
    dow = dates[i].dayofweek
    if dow <= 4: weekly[i] = 30  # Weekday (elective + emergency)
    else: weekly[i] = -40  # Weekend (emergency only)

flu = np.zeros(N_POINTS)
for i in range(N_POINTS):
    m = dates[i].month
    if m <= 3: flu[i] = 50
    elif m == 12: flu[i] = 30

holiday = np.zeros(N_POINTS)
for i in range(N_POINTS):
    m, d = dates[i].month, dates[i].day
    if (m == 12 and d >= 24) or (m == 1 and d <= 2): holiday[i] = -60
    elif m == 11 and 22 <= d <= 28: holiday[i] = -40

noise = np.random.normal(0, 15, N_POINTS)
admissions = base + weekly + flu + holiday + noise
admissions = np.maximum(admissions, 50).round(0).astype(int)

df = pd.DataFrame({'date': dates, 'admissions': admissions})
print(f"Dataset shape: {df.shape}")
df.head()

Dataset shape: (730, 2)


,date,admissions
0,2022-01-01,157
1,2022-01-02,148
2,2022-01-03,290
3,2022-01-04,303
4,2022-01-05,277


## Data Validation Checks

In [5]:
assert df.isnull().sum().sum() == 0, "Missing values"
assert (df[TARGET] > 0).all(), "Non-positive values"
assert df['date'].is_monotonic_increasing, "Not sorted"
assert len(df) == N_POINTS, "Row count mismatch"
print("All validation checks passed.")

All validation checks passed.


## Exploratory Data Analysis

In [6]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
axes[0, 0].plot(df['date'], df[TARGET], linewidth=0.6)
axes[0, 0].set_title('admissions Over Time')
axes[0, 1].hist(df[TARGET], bins=30, edgecolor='black', alpha=0.7)
axes[0, 1].set_title('Distribution')
from pandas.plotting import autocorrelation_plot
autocorrelation_plot(df[TARGET], ax=axes[1, 0])
axes[1, 0].set_xlim(0, min(60, N_POINTS // 2))
axes[1, 0].set_title('Autocorrelation')
rw = min(SEASON_LENGTH * 2, N_POINTS // 4)
axes[1, 1].plot(df['date'], df[TARGET].rolling(rw).mean())
axes[1, 1].set_title(f'Rolling {rw}-period Mean')
plt.tight_layout()
plt.savefig('eda.png', dpi=100, bbox_inches='tight')
plt.show()
print("EDA complete.")

EDA complete.


## Target Analysis

In [7]:
print("admissions Statistics:")
print(df[TARGET].describe())
print(f"\nCV: {df[TARGET].std() / df[TARGET].mean():.3f}")
print(f"Skewness: {df[TARGET].skew():.3f}")

admissions Statistics:
count    730.000000
mean     276.795890
std       47.768253
min      148.000000
25%      245.000000
50%      278.000000
75%      313.000000
max      392.000000
Name: admissions, dtype: float64

CV: 0.173
Skewness: -0.253


## Train / Validation / Test Split Strategy

Time-aware split (no shuffling):
- **Train**: all except last 28
- **Validation**: 14 points
- **Test**: last 14 points

In [8]:
train = df.iloc[:-(VAL_SIZE + TEST_SIZE)].copy()
val = df.iloc[-(VAL_SIZE + TEST_SIZE):-TEST_SIZE].copy()
test = df.iloc[-TEST_SIZE:].copy()
print(f"Train: {len(train)}, Val: {len(val)}, Test: {len(test)}")

Train: 702, Val: 14, Test: 14


## Preprocessing Strategy

Minimal preprocessing — tree models handle raw features. Features created next.

In [9]:
df_full = df.copy().sort_values('date').reset_index(drop=True)
print('Data ready.')

Data ready.


## Feature Engineering

In [10]:
def create_features(data):
    d = data.copy()
    d['dow'] = d['date'].dt.dayofweek
    d['month'] = d['date'].dt.month
    d['day'] = d['date'].dt.day
    d['week_of_year'] = d['date'].dt.isocalendar().week.astype(int)
    for lag in [1, 7, 14, 28]:
        d[f'lag_{lag}'] = d[TARGET].shift(lag)
    for w in [7, 14, 28]:
        d[f'rmean_{w}'] = d[TARGET].shift(1).rolling(w).mean()
        d[f'rstd_{w}'] = d[TARGET].shift(1).rolling(w).std()
    return d

df_feat = create_features(df_full).dropna().reset_index(drop=True)
feature_cols = [c for c in df_feat.columns if c not in ['date', TARGET]]
print(f"Features: {len(feature_cols)} columns, {len(df_feat)} rows")

Features: 14 columns, 702 rows


## Baseline Model — Naive Forecasts

In [11]:
def calc_metrics(y_true, y_pred, name=""):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = np.mean(np.abs((y_true - y_pred) / np.maximum(y_true, 1))) * 100
    print(f"{name:30s} | MAE: {mae:10.1f} | RMSE: {rmse:10.1f} | MAPE: {mape:5.2f}%")
    return {'model': name, 'MAE': mae, 'RMSE': rmse, 'MAPE': mape}

results = []
naive_pred = np.full(TEST_SIZE, train[TARGET].iloc[-1])
results.append(calc_metrics(test[TARGET].values, naive_pred, "Naive (Last Value)"))

src = df_full[TARGET].values
ti = len(df_full) - TEST_SIZE
sn_pred = src[ti - SEASON_LENGTH:ti - SEASON_LENGTH + TEST_SIZE]
results.append(calc_metrics(test[TARGET].values, sn_pred, f"Seasonal Naive ({SEASON_LENGTH})"))

Naive (Last Value)             | MAE:       42.9 | RMSE:       53.5 | MAPE: 15.12%
Seasonal Naive (7)             | MAE:       34.8 | RMSE:       44.8 | MAPE: 12.48%


## LazyPredict Benchmark (Lag-Feature Tabular)

In [12]:
from lazypredict.Supervised import LazyRegressor

ct = len(df_feat) - TEST_SIZE
cv = ct - VAL_SIZE
X_tr = df_feat.iloc[:cv][feature_cols]
y_tr = df_feat.iloc[:cv][TARGET]
X_va = df_feat.iloc[cv:ct][feature_cols]
y_va = df_feat.iloc[cv:ct][TARGET]

reg = LazyRegressor(verbose=0, ignore_warnings=True, custom_metric=None, predictions=True)
lp_models, lp_preds = reg.fit(X_tr, X_va, y_tr, y_va)
print("\nLazyPredict Top 10:")
print(lp_models.head(10).to_string())


LazyPredict Top 10:
                          Adjusted R-Squared  R-Squared        RMSE  Time Taken
Model                                                                          
GaussianProcessRegressor         1205.345925 -91.641994  319.381445    0.074075
KernelRidge                       971.680992 -73.667769  286.729463    0.062112
DummyRegressor                     69.719031  -4.286079   76.290871    0.010620
MLPRegressor                       67.215954  -4.093535   74.888542    0.402917
QuantileRegressor                  65.876631  -3.990510   74.127304    0.082731
NuSVR                              45.488224  -2.422171   61.384199    0.032068
SVR                                39.826730  -1.986672   57.345516    0.028793
DecisionTreeRegressor              29.671949  -1.205535   49.279089    0.017045
ExtraTreeRegressor                 25.093444  -0.853342   45.173475    0.014271
XGBRegressor                       24.145700  -0.780438   44.276085    2.478505


## FLAML AutoML Run

In [13]:
from flaml import AutoML

automl = AutoML()
automl.fit(
    X_train=X_tr, y_train=y_tr,
    task='regression', time_budget=30, metric='mae',
    estimator_list=['lgbm', 'rf', 'extra_tree', 'catboost'],
    seed=SEED, verbose=0
)
print(f"FLAML best: {automl.best_estimator}")
flaml_val = automl.predict(X_va)
results.append(calc_metrics(y_va.values, flaml_val, f"FLAML ({automl.best_estimator})"))

X_te = df_feat.iloc[ct:][feature_cols]
y_te = df_feat.iloc[ct:][TARGET]
flaml_test = automl.predict(X_te)
results.append(calc_metrics(y_te.values, flaml_test, f"FLAML Test ({automl.best_estimator})"))

FLAML best: catboost
FLAML (catboost)               | MAE:       34.8 | RMSE:       39.0 | MAPE:  9.83%
FLAML Test (catboost)          | MAE:       27.7 | RMSE:       33.8 | MAPE:  9.35%


## StatsForecast — Statistical Forecasting

In [14]:
from statsforecast import StatsForecast
from statsforecast.models import AutoARIMA, AutoETS, SeasonalNaive as SFSeasonalNaive

sf_df = df_full[['date', TARGET]].copy()
sf_df.columns = ['ds', 'y']
sf_df['unique_id'] = 'series1'
sf_df = sf_df[['unique_id', 'ds', 'y']]

sf_train = sf_df.iloc[:-TEST_SIZE]
sf = StatsForecast(
    models=[AutoARIMA(season_length=SEASON_LENGTH), AutoETS(season_length=SEASON_LENGTH),
            SFSeasonalNaive(season_length=SEASON_LENGTH)],
    freq=FREQ, n_jobs=1
)
sf.fit(sf_train)
sf_preds = sf.predict(h=TEST_SIZE).reset_index()

y_test_sf = test[TARGET].values
for col in ['AutoARIMA', 'AutoETS', 'SeasonalNaive']:
    if col in sf_preds.columns:
        results.append(calc_metrics(y_test_sf, sf_preds[col].values, f"SF {col}"))
print("StatsForecast complete.")

SF AutoARIMA                   | MAE:       34.3 | RMSE:       42.2 | MAPE: 12.76%
SF AutoETS                     | MAE:       41.3 | RMSE:       51.2 | MAPE: 15.31%
SF SeasonalNaive               | MAE:       43.3 | RMSE:       53.7 | MAPE: 16.03%
StatsForecast complete.


## Top 3 Model Selection

In [15]:
results_df = pd.DataFrame(results).sort_values('MAE').reset_index(drop=True)
print("All Results:")
print(results_df.to_string(index=False))
print("\nTop 3:")
print(results_df.head(3).to_string(index=False))

All Results:
                model       MAE      RMSE      MAPE
FLAML Test (catboost) 27.703481 33.842912  9.349626
         SF AutoARIMA 34.335579 42.246612 12.759502
   Seasonal Naive (7) 34.785714 44.790783 12.482045
     FLAML (catboost) 34.831384 39.030742  9.830251
           SF AutoETS 41.299000 51.225847 15.312831
   Naive (Last Value) 42.857143 53.454921 15.122630
     SF SeasonalNaive 43.285713 53.666962 16.034527

Top 3:
                model       MAE      RMSE      MAPE
FLAML Test (catboost) 27.703481 33.842912  9.349626
         SF AutoARIMA 34.335579 42.246612 12.759502
   Seasonal Naive (7) 34.785714 44.790783 12.482045


## Final Training and Evaluation of Top 3

In [16]:
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(test['date'].values, test[TARGET].values, 'ko-', label='Actual', ms=4)
ax.plot(test['date'].values, flaml_test, 's--', label=f'FLAML ({automl.best_estimator})', ms=4)
for col in ['AutoARIMA', 'AutoETS']:
    if col in sf_preds.columns:
        ax.plot(test['date'].values[:len(sf_preds)], sf_preds[col].values, 'o--', label=f'SF {col}', ms=4)
ax.set_title('Forecast vs Actual — Test Set')
ax.legend()
plt.tight_layout()
plt.savefig('forecast_comparison.png', dpi=100, bbox_inches='tight')
plt.show()

## Error Analysis

In [17]:
errors = test[TARGET].values - flaml_test
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar(range(len(errors)), errors, alpha=0.7)
axes[0].set_title('Residuals (Actual - FLAML)')
axes[0].axhline(y=0, color='r', linestyle='--')
axes[1].hist(errors, bins=10, edgecolor='black', alpha=0.7)
axes[1].set_title('Residual Distribution')
plt.tight_layout()
plt.savefig('error_analysis.png', dpi=100, bbox_inches='tight')
plt.show()
print(f"Mean residual: {np.mean(errors):.2f}, Std: {np.std(errors):.2f}")

Mean residual: 6.46, Std: 33.22


## Interpretation and Business Insight

- Weekday admissions are 30-40% higher than weekends (elective procedures)
- Flu season (Jan-Mar) increases volume by ~20%
- Holiday dips require adjusted staffing
- Growth trend suggests increasing capacity needs
- Statistical models capture seasonality well

## Limitations

1. Synthetic — real hospitals have case-mix complexity
2. No admission type (elective vs emergency)
3. No disease-specific modeling
4. No capacity constraint feedback (diversion effects)

## How to Improve This Project

1. Segment by admission type and department
2. Add flu/COVID surveillance data as features
3. Model length-of-stay for census forecasting
4. Add ED arrival forecasts as leading indicator
5. Use hierarchical forecasting across departments

## Production Considerations

- Daily retraining with latest data
- Integration with bed management systems
- Dashboard showing predicted vs actual census
- Alert on surge conditions (>90% capacity predicted)

## Common Mistakes

1. Not separating elective from emergency admissions
2. Ignoring flu season in feature design
3. Over-fitting on holiday patterns with few examples
4. Not accounting for day-of-week effects

## Mini Challenge / Exercises

1. Add flu surveillance data and measure improvement
2. Forecast separately for elective vs. emergency
3. Combine admission forecast with average LOS for census prediction
4. Simulate surge capacity trigger rules

## Final Summary / Key Takeaways

1. Hospital admission forecasting is critical for capacity planning
2. Weekly and seasonal (flu) patterns drive most variation
3. Holiday dips require proactive schedule adjustments
4. 14-day horizons enable staffing and OR scheduling
5. Statistical models handle the seasonal structure well

In [18]:
import json
metrics = {
    'project': 'Hospital Admission Forecasting',
    'best_model': results_df.iloc[0]['model'],
    'best_MAE': float(results_df.iloc[0]['MAE']),
    'best_RMSE': float(results_df.iloc[0]['RMSE']),
    'best_MAPE': float(results_df.iloc[0]['MAPE']),
    'all_results': results_df.to_dict('records')
}
with open('metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)
print("Metrics saved.")
print("\n=== Hospital Admission Forecasting — Complete ===")

Metrics saved.

=== Hospital Admission Forecasting — Complete ===
